In [1]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama


# Router le return garne output ko schema
class RouteQuery(BaseModel):
    """User ko question kun datasource ma pathaune bhanne define gareko."""

    datasource: Literal[
        "python_docs",
        "js_docs",
        "golang_docs",
    ] = Field(description="Question ko lagi sabai bhanda relevant datasource.")


# Local Ollama LLM
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Structured output use gareko
structured_llm = llm.with_structured_output(RouteQuery)


# Router ko system prompt
system = """
You are an expert at routing user questions.

Choose the most appropriate datasource based on the programming language mentioned in the question.

Available datasources:
- python_docs
- js_docs
- golang_docs

Return only the datasource.
"""


# Prompt template create gareko
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)


# Router chain create gareko
router = prompt | structured_llm

In [2]:
# Routing test garna sample question
question = """
Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    ["human", "speak in {language}"]
)

prompt.invoke("french")
"""


# Router execute gareko
result = router.invoke(
    {
        "question": question,
    }
)


# Router le choose gareko datasource print garne
print(result)

datasource='python_docs'


In [3]:
result

RouteQuery(datasource='python_docs')

In [4]:
result.datasource

'python_docs'

In [5]:
from langchain_core.runnables import RunnableLambda


# Router le return gareko datasource anusar chain choose garne function
def choose_route(result):

    # Python related question bhaye
    if "python_docs" in result.datasource.lower():
        # Python RAG chain return garne
        return "chain for python_docs"

    # JavaScript related question bhaye
    elif "js_docs" in result.datasource.lower():
        # JavaScript RAG chain return garne
        return "chain for js_docs"

    # Aru sabai case ma Go documentation use garne
    else:
        return "chain for golang_docs"


# Router ko output anusar appropriate chain select garne
full_chain = router | RunnableLambda(choose_route)

In [6]:
# Full routing chain execute gareko
response = full_chain.invoke(
    {
        "question": question,
    }
)


# Router le choose gareko chain ko result print garne
print(response)

chain for python_docs
